# UC4 — Asset duplicate refusal: config round-trip

**Persona:** ingestion-pipeline operator who needs idempotent uploads
by default but switches to fail-on-duplicate when a strict batch is
running.

**Goal:** demonstrate that `assets_write_policy.on_conflict` is a real
config-API surface — not a baked-in default.  We toggle it through two
PATCH cycles and observe both behaviors:

1. PATCH `on_conflict=refuse_return` → re-uploading the same asset_id
   returns 200 with the existing row (idempotent).
2. PATCH back to `on_conflict=refuse_fail` (the default) → re-upload
   returns **409 Conflict**.

Both modes use `[asset_id, filename]` as the identity matcher chain.

**Critical:** `assets_write_policy` is a **catalog-scope** config (not
collection-scope) — it controls assets across every collection in the
catalog.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"cf_uc4_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
if not TOKEN:
    print("\nWARNING: no Bearer token set — config writes will 401.")
    print("Set DYNASTORE_TOKEN before running write cells.")


In [ ]:
# Create catalog (idempotent: 409 = already exists)
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": f"Cycle F UC walkthrough {RUN_ID}",
    "description": "Ephemeral catalog for cycle_f_use_cases notebook.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
if r.status_code in (200, 201):
    print(f"Catalog '{CATALOG_ID}' created.")
elif r.status_code == 409:
    print(f"Catalog '{CATALOG_ID}' already exists.")
else:
    raise RuntimeError(f"Catalog create failed: {r.status_code}: {r.text}")

# Create collection (idempotent)
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": f"UC collection {RUN_ID}",
    "description": "Walkthrough collection — defaults inherited then PATCHed.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
if r.status_code in (200, 201):
    print(f"Collection '{COLLECTION_ID}' created.")
elif r.status_code == 409:
    print(f"Collection '{COLLECTION_ID}' already exists.")
else:
    raise RuntimeError(f"Collection create failed: {r.status_code}: {r.text}")


In [ ]:
# Helper — show explicit + waterfall-resolved view for a plugin.
#
# The configs API exposes:
#   GET /configs/.../plugins/{plugin_id}                  → explicit (per-scope)
#   GET /configs/.../plugins/{plugin_id}?resolved=true    → waterfall-resolved
# There is NO `/effective` endpoint — the resolved form is reached via
# the `?resolved=true` query string.
def show_config_delta(plugin_id: str, level: str = "collection") -> None:
    if level == "collection":
        base = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    elif level == "catalog":
        base = f"/configs/catalogs/{CATALOG_ID}"
    else:
        raise ValueError(level)
    rx = client.get(f"{base}/plugins/{plugin_id}")
    rr = client.get(f"{base}/plugins/{plugin_id}", params={"resolved": "true"})
    print(f"\n=== {plugin_id} @ {level} ===")
    print("EXPLICIT (only fields stored at this scope):")
    if rx.status_code == 200:
        print(json.dumps(rx.json(), indent=2)[:600])
    else:
        print(f"  ({rx.status_code} — none stored, every field inherited)")
    print("RESOLVED (waterfall over platform → catalog → collection):")
    if rr.status_code == 200:
        print(json.dumps(rr.json(), indent=2)[:800])
    else:
        print(f"  ({rr.status_code}) {rr.text[:160]}")


In [ ]:
def put_config(plugin_id: str, body: dict, level: str = "collection") -> None:
    if level == "collection":
        url = f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}"
    else:
        url = f"/configs/catalogs/{CATALOG_ID}/plugins/{plugin_id}"
    r = client.put(url, content=json.dumps(body), timeout=60.0)
    print(f"PUT {plugin_id}: {r.status_code}")
    if r.status_code not in (200, 201, 204):
        print(f"  body: {r.text[:300]}")
        if r.status_code == 401:
            raise RuntimeError("Unauthorized — set DYNASTORE_TOKEN.")
        raise RuntimeError(f"PUT failed: {r.status_code}")


## Step 1 — PATCH catalog-scope `asset_write_policy_defaults` to `refuse_return`

`AssetWritePolicyDefaults` is the posture-only class at platform/
catalog tier; it cascades down to all owned collections.  The full
matcher-bearing class `AssetsWritePolicy` lives at collection tier
(`assets_write_policy`).  Either works for this UC; the defaults
class is idiomatic for catalog-wide posture changes.


In [ ]:
relaxed_policy = {
    "on_conflict": "refuse_return",
    "require_filename": False,
}
put_config("asset_write_policy_defaults", relaxed_policy, level="catalog")
show_config_delta("asset_write_policy_defaults", level="catalog")


## Step 2 — Register + re-register an asset under `refuse_return`

Asset records are registered via
`POST /assets/catalogs/{cat}/collections/{coll}` with a JSON body
(no multipart upload — the binary, when there is one, lives in GCS
and is registered automatically via OBJECT_FINALIZE pub/sub).

First create: 201.  Second create with same `asset_id` under
`refuse_return`: 200 with the existing row (idempotent).


In [ ]:
ASSET_ID = "feature-pack-A"
register_url = f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
body = {
    "asset_id": ASSET_ID,
    "asset_type": "ASSET",
    "uri": f"file:///fixtures/{ASSET_ID}.geojson",
    "metadata": {"source": f"uc4-{RUN_ID}"},
}

r1 = client.post(register_url, content=json.dumps(body))
print(f"first register : {r1.status_code} — {r1.text[:200]}")
assert r1.status_code in (200, 201), f"first register failed: {r1.status_code}"

r2 = client.post(register_url, content=json.dumps(body))
print(f"second register: {r2.status_code} — {r2.text[:200]}")
# refuse_return → idempotent 200
assert r2.status_code in (200, 201), \
    f"refuse_return should be idempotent (200), got {r2.status_code}"
print("  idempotent: refuse_return returned existing row, no error.")


## Step 3 — PATCH back to `refuse_fail` (the platform default)


In [ ]:
strict_policy = {
    "on_conflict": "refuse_fail",
    "require_filename": False,
}
put_config("asset_write_policy_defaults", strict_policy, level="catalog")
show_config_delta("asset_write_policy_defaults", level="catalog")


## Step 4 — Re-register under `refuse_fail` → 409 Conflict


In [ ]:
r3 = client.post(register_url, content=json.dumps(body))
print(f"third register : {r3.status_code} — {r3.text[:200]}")
assert r3.status_code == 409, \
    f"refuse_fail should return 409 on duplicate, got {r3.status_code}"
print("  refuse_fail confirmed: duplicate asset_id returned 409 Conflict.")


## Teardown

In [ ]:
# Teardown — delete the ephemeral catalog. Comment out to keep state.
r = client.delete(
    f"/stac/catalogs/{CATALOG_ID}",
    params={"force": "true"},
    timeout=60.0,
)
print(f"teardown DELETE /stac/catalogs/{CATALOG_ID}: {r.status_code}")
client.close()
